Analisa saída do pipeline AVeriTeC (JSON). Calcula métricas básicas, checa inconsistências e dá interpretação com baseline.

In [3]:
import json, pandas as pd, numpy as np

path = '../outputs/2026-03-22_01-05-50/averitec_dev.json'
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

len(data)

1

In [4]:
def normalize_label(x):
    if x is None: return None
    x = str(x).strip().upper()
    if x in ['SUPPORTED','SUPPORT']: return 'SUPPORTED'
    if x in ['REFUTED','REFUTE']: return 'REFUTED'
    if 'NOT' in x: return 'NOT ENOUGH EVIDENCE'
    if 'CONFLICT' in x: return 'CONFLICTING EVIDENCE/CHERRYPICKING'
    return x

rows = []
for ex in data:
    pred = normalize_label(ex.get('prediction'))
    gold = normalize_label(ex.get('gold_label'))
    steps = ex.get('pipeline', {}).get('steps', [])
    evid_counts = [len(s.get('evidence', [])) for s in steps]
    rows.append({
        'claim': ex.get('claim'),
        'prediction': pred,
        'gold': gold,
        'correct': int(pred == gold) if pred and gold else np.nan,
        'n_steps': len(steps),
        'avg_evidence_per_step': np.mean(evid_counts) if evid_counts else 0.0
    })

df = pd.DataFrame(rows)
df.head()

,claim,prediction,gold,correct,n_steps,avg_evidence_per_step
0,"In a letter to Steve Jobs, Sean Connery refuse...",REFUTED,REFUTED,1,2,4.0


In [5]:
accuracy = df['correct'].mean()

# proxy simples: mistura acerto + evidência média (normalizada)
evidence_score = df['avg_evidence_per_step'].mean()
proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

accuracy, evidence_score, proxy_score

(np.float64(1.0), np.float64(4.0), np.float64(0.94))

In [6]:
baseline = {
    'accuracy': {'random': 0.33, 'baseline': 0.5, 'strong': 0.7},
    'proxy_score': {'random': 0.15, 'baseline': 0.4, 'strong': 0.65}
}

def interpret(v, m):
    ref = baseline[m]
    if v < ref['random']: return 'ruim'
    if v < ref['baseline']: return 'fraco'
    if v < ref['strong']: return 'ok'
    return 'forte'

print('accuracy:', round(accuracy,3), '->', interpret(accuracy,'accuracy'))
print('proxy:', round(proxy_score,3), '->', interpret(proxy_score,'proxy_score'))

accuracy: 1.0 -> forte
proxy: 0.94 -> forte


In [7]:
errors = []

# casos onde evidência contradiz label final
for ex in data:
    pred = normalize_label(ex.get('prediction'))
    steps = ex.get('pipeline', {}).get('steps', [])
    for s in steps:
        for ev in s.get('evidence', []):
            if 'label' in ev and pred and ev['label'] != pred:
                errors.append('evidence-label mismatch')

len(errors)

0

In [8]:
df_errors = df[df['correct'] == 0]
df_errors[['claim','prediction','gold']].head(10)

,claim,prediction,gold
